# Experiment 3: Circuit Discovery & Span Validation

**Criterion 3**: Within-span similarity elevation > population mean + 1σ
**Criterion 4**: Circuit diversity ≥ 60% layer coverage
**Criterion 5**: Class purity distribution is bimodal

Runs the full span-centric circuit discovery pipeline, validates discovered circuits,
and checks multi-circuit membership.

## Colab Setup
Clones the repo, installs dependencies, and mounts Google Drive.

In [ ]:
import os, sys

# 1. Clone repository
REPO_DIR = '/content/trainable-circuits'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/jacobposchl/trainable-circuits {REPO_DIR}

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# 2. Install extra dependencies
!pip install hdbscan umap-learn -q

# 3. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 4. Path constants
DRIVE_DIR  = '/content/drive/MyDrive/ctls'
DATA_DIR   = DRIVE_DIR + '/data/raw'
CKPT_DIR   = DRIVE_DIR + '/experiments'
CONFIG_DIR = REPO_DIR  + '/configs'

print('Repo:     ', REPO_DIR)
print('Data:     ', DATA_DIR)
print('Checkpts: ', CKPT_DIR)

In [ ]:
import torch
import yaml
import numpy as np
import matplotlib.pyplot as plt
from models.backbone import FrozenBackbone
from models.meta_encoder import MetaEncoder
from data.cifar import get_standard_loaders
from evaluation.circuit_analysis import CircuitAnalyzer
from evaluation.discovery import SpanCentricDiscovery
from evaluation.metrics import (
    within_span_elevation,
    circuit_diversity,
    class_purity_distribution,
)
from evaluation.circuit_viz import (
    plot_span_coverage,
    plot_multi_circuit_histogram,
    plot_circuit_members,
)
from evaluation.circuit_analysis import denormalize

In [ ]:
# Load config and override data directory
config_path     = CONFIG_DIR + '/phase1.yaml'
checkpoint_path = CKPT_DIR  + '/phase1/best.pt'

with open(config_path) as f:
    config = yaml.safe_load(f)

config['data']['data_dir'] = DATA_DIR

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
# Build models and load weights
flow_cfg = config['model'].get('flow_compression', {})

backbone = FrozenBackbone(
    arch=config['model']['arch'],
    num_classes=config['model']['num_classes'],
    pretrained=config['model']['pretrained'],
    grid_size=flow_cfg.get('grid_size', 4),
    flow_dim=flow_cfg.get('flow_dim', 256),
).to(device)

meta_encoder = MetaEncoder(
    layer_dims=backbone.layer_dims,
    projection_dim=config['model']['meta_encoder']['projection_dim'],
    n_heads=config['model']['meta_encoder']['n_heads'],
    n_transformer_layers=config['model']['meta_encoder']['n_transformer_layers'],
    dropout=config['model']['meta_encoder']['dropout']
).to(device)

ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
meta_encoder.load_state_dict(ckpt['meta_encoder_state'])
meta_encoder.eval()
print('Model loaded.')

In [ ]:
# Create validation loader and collect representations
_, val_loader = get_standard_loaders(
    data_dir=DATA_DIR,
    batch_size=config['data'].get('batch_size', 256),
    num_workers=0,
    augment=False,
    download=False,
)

analyzer = CircuitAnalyzer(backbone, meta_encoder, val_loader, device)
data     = analyzer.collect_representations(max_samples=2000)
N = data['labels'].shape[0]
L = len(data['z_list'])
print(f'Collected {N} samples, {L} layers')

# Convert z_list to numpy — one [N, d] array per layer
z_list_np  = [z.numpy() for z in data['z_list']]
labels_np  = data['labels'].numpy()

In [ ]:
# Run image-centric circuit discovery: UMAP + HDBSCAN per span
disc_cfg  = config['discovery']
discovery = SpanCentricDiscovery(
    n_layers=L,
    projection_dim=config['model']['meta_encoder']['projection_dim'],
    umap_n_components=disc_cfg.get('umap_n_components', 15),
    umap_n_neighbors=disc_cfg.get('umap_n_neighbors', 15),
    min_cluster_fraction=disc_cfg['min_cluster_fraction'],
    max_cluster_fraction=disc_cfg.get('max_cluster_fraction', 0.40),
    min_cluster_size=disc_cfg['hdbscan_min_cluster_size'],
)

circuits = discovery.discover_all(z_list_np)
print(f'Discovered {len(circuits)} canonical circuits')

In [ ]:
# Criterion 3: Within-span similarity elevation
# For each circuit, compare mean pairwise z-similarity within the cluster
# to the population of all pairs at that span.
for i, circuit in enumerate(circuits):
    pop_sims     = discovery.compute_span_similarities(z_list_np, circuit['span'])
    cluster_sims = discovery.compute_span_similarities(z_list_np, circuit['span'],
                                                        image_mask=circuit['image_mask'])
    result = within_span_elevation(cluster_sims, pop_sims)
    status = '\u2713' if result['passes'] else '\u2717'
    print(f"Circuit {i} span {circuit['span']} (n={circuit['size']}): "
          f"elevation = {result['elevation_sigma']:.2f}\u03c3  {status}")

In [ ]:
# Criterion 4: Circuit diversity
# circuit_diversity expects list of (l_start, l_end) spans
spans  = [c['span'] for c in circuits]
result = circuit_diversity(spans, L)
print(f"Layer coverage: {result['coverage']:.1%}  "
      f"(target \u2265 60%, {'PASS' if result['passes'] else 'FAIL'})")

fig = plot_span_coverage(circuits, L)
plt.tight_layout()
plt.show()

In [ ]:
# Criterion 5: Class purity distribution
purities = [
    SpanCentricDiscovery.compute_class_purity(c, labels_np)
    for c in circuits
]

result = class_purity_distribution(purities)
print(f"Agnostic (<0.3): {result['n_agnostic']}, "
      f"Specific (>0.7): {result['n_specific']}  "
      f"({'PASS' if result['passes'] else 'FAIL'})")

In [ ]:
# Multi-circuit membership distribution (per image)
counts = SpanCentricDiscovery.multi_circuit_membership(circuits, n_images=N)

fig = plot_multi_circuit_histogram(counts)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize example circuits (first 3)
# For each circuit, show images in the cluster with their mean z-similarity
# profile (mean similarity to all other circuit members at each layer).
for i, circuit in enumerate(circuits[:3]):
    show_idx = np.where(circuit['image_mask'])[0][:16]
    n_show   = len(show_idx)

    # Per-image mean z-similarity profile across layers
    input_profiles = np.zeros((n_show, L))
    for l in range(L):
        z_show    = z_list_np[l][show_idx]                            # [n_show, d]
        z_circuit = z_list_np[l][circuit['image_mask']]               # [n_circuit, d]
        input_profiles[:, l] = (z_show @ z_circuit.T).mean(axis=1)   # mean sim to cluster

    imgs_np = denormalize(data['images'][show_idx]).numpy()
    lbls_np = labels_np[show_idx]

    fig = plot_circuit_members(imgs_np, lbls_np, input_profiles, span=circuit['span'])
    plt.suptitle(f"Circuit {i}  span={circuit['span']}  n={circuit['size']}", y=1.01)
    plt.tight_layout()
    plt.show()